In [21]:

from pathlib import Path
import pandas as pd

DATA_DIR = Path("data/raw")
AVAILABLE = {
    "car": {"file": "car.csv", "target": "class"},
    "aps": {"file": "aps_training.csv", "target": "class", "na_values": ["na"]},
    "covertype": {"file": "covertype.csv", "target": "Cover_Type"},
    "jannis": {"file": "jannis.csv", "target": "class"},
}

def _exists(name): return (DATA_DIR / AVAILABLE[name]["file"]).exists()

print("Available files:")
for k in AVAILABLE: 
    print(f" - {k:10s} -> {AVAILABLE[k]['file']}  {'✓' if _exists(k) else '✗'}")


Available files:
 - car        -> car.csv  ✓
 - aps        -> aps_training.csv  ✓
 - covertype  -> covertype.csv  ✓
 - jannis     -> jannis.csv  ✓


In [22]:
DATASET = "aps"

In [25]:

# Load selected dataset
from IPython.display import display
import pandas as pd

name = DATASET
info = AVAILABLE[name]
path = (DATA_DIR / info["file"])
read_kwargs = {}
print(info)
if "na_values" in info: 
    read_kwargs["na_values"] = info["na_values"]

df = pd.read_csv(path, **read_kwargs)
print(f"Loaded: {name} -> {path.name}")
print("Shape:", df.shape)
print("Memory MB (approx):", round(df.memory_usage(index=True, deep=True).sum()/1e6, 2))
print("Columns sample:", list(df.columns)[:12], "..." if df.shape[1] > 12 else "")
display(df.head())


{'file': 'aps_training.csv', 'target': 'class', 'na_values': ['na']}
Loaded: aps -> aps_training.csv
Shape: (60000, 171)
Memory MB (approx): 85.2
Columns sample: ['class', 'aa_000', 'ab_000', 'ac_000', 'ad_000', 'ae_000', 'af_000', 'ag_000', 'ag_001', 'ag_002', 'ag_003', 'ag_004'] ...


,class,aa_000,ab_000,ac_000,ad_000,ae_000,af_000,ag_000,ag_001,ag_002,...,ee_002,ee_003,ee_004,ee_005,ee_006,ee_007,ee_008,ee_009,ef_000,eg_000
0,neg,76698,NaN,2.130706e+09,280.0,0.0,0.0,0.0,0.0,0.0,...,1240520.0,493384.0,721044.0,469792.0,339156.0,157956.0,73224.0,0.0,0.0,0.0
1,neg,33058,NaN,0.000000e+00,NaN,0.0,0.0,0.0,0.0,0.0,...,421400.0,178064.0,293306.0,245416.0,133654.0,81140.0,97576.0,1500.0,0.0,0.0
2,neg,41040,NaN,2.280000e+02,100.0,0.0,0.0,0.0,0.0,0.0,...,277378.0,159812.0,423992.0,409564.0,320746.0,158022.0,95128.0,514.0,0.0,0.0
3,neg,12,0.0,7.000000e+01,66.0,0.0,10.0,0.0,0.0,0.0,...,240.0,46.0,58.0,44.0,10.0,0.0,0.0,0.0,4.0,32.0
4,neg,60874,NaN,1.368000e+03,458.0,0.0,0.0,0.0,0.0,0.0,...,622012.0,229790.0,405298.0,347188.0,286954.0,311560.0,433954.0,1218.0,0.0,0.0


In [26]:

# Split to X, y (with target detection fallback)
target_col = AVAILABLE[name].get("target")
if target_col not in df.columns:
    for cand in ["class","target","Cover_Type","label"]:
        if cand in df.columns: 
            target_col = cand 
            print(f"[info] Using guessed target: {target_col}")
            break
    else:
        target_col = df.columns[-1]
        print(f"[warn] Falling back to last column as target: {target_col}")

y = df[target_col]
X = df.drop(columns=[target_col])

print(f"Target column: {target_col}")
print("Class distribution (top 10):")
print(y.value_counts().head(10))
print("\nX shape:", X.shape, " | y shape:", y.shape)


Target column: class
Class distribution (top 10):
class
neg    59000
pos     1000
Name: count, dtype: int64

X shape: (60000, 170)  | y shape: (60000,)


In [27]:

# Quick dtypes + missingness snapshot
import numpy as np
print("Feature dtype counts:")
print(X.dtypes.astype(str).value_counts())
print("\nTop-10 columns by missingness:")
print(X.isna().mean().sort_values(ascending=False).head(10))

cat_cols = [c for c in X.columns if X[c].dtype == 'object']
if cat_cols:
    print("\nSample categorical cardinalities:")
    for c in cat_cols:
        print(f"  - {c}: {X[c].nunique()} unique")


Feature dtype counts:
float64    169
int64        1
Name: count, dtype: int64

Top-10 columns by missingness:
br_000    0.821067
bq_000    0.812033
bp_000    0.795667
bo_000    0.772217
ab_000    0.772150
cr_000    0.772150
bn_000    0.733483
bm_000    0.659150
bl_000    0.454617
bk_000    0.383900
dtype: float64


In [ ]:
# Per-feature histograms: categorical => bar counts, numeric => 50 bins [min, max]
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pandas.api.types import is_numeric_dtype

assert 'X' in globals() and 'name' in globals(), "Run the previous cells to define X and dataset 'name'."
outdir = Path("plots")/name
outdir.mkdir(parents=True, exist_ok=True)

for col in X.columns:
    s = X[col]
    fig, ax = plt.subplots(figsize=(8, 4.5))  # one chart per figure (no subplots)
    if is_numeric_dtype(s):
        vals = s.dropna().to_numpy()
        if vals.size == 0:
            ax.text(0.5, 0.5, "All values are NaN", ha='center', va='center')
        else:
            lo, hi = float(np.nanmin(vals)), float(np.nanmax(vals))
            if lo == hi:  # constant column
                ax.hist(vals, bins=1)
            else:
                bins = np.linspace(lo, hi, 51)  # 50 bins from min to max
                ax.hist(vals, bins=bins)
        ax.set_title(f"{col} — numeric histogram (50 bins min→max)")
        ax.set_xlabel(col)
        ax.set_ylabel("count")
    else:
        counts = (s.fillna("(NA)").astype(str)).value_counts(dropna=False)
        ax.bar(counts.index, counts.values)
        ax.set_title(f"{col} — categorical histogram")
        ax.set_xlabel(col) 
        ax.set_ylabel("count")
        ax.set_xticks(range(len(counts.index)))
        ax.set_xticklabels(counts.index, rotation=90, fontsize=8)
    plt.tight_layout()
    png_path = outdir/f"{col}.png"
    fig.savefig(png_path, dpi=120)
    plt.show()
    print(f"saved: {png_path}")


In [29]:
# Missing-value ratios per feature (+ dtype, n_missing, n_unique); saves CSV to reports/
import pandas as pd
from pathlib import Path
from IPython.display import display

assert 'X' in globals() and 'name' in globals(), "Run previous cells first."

Path("reports").mkdir(exist_ok=True)

miss_df = pd.DataFrame({
    "dtype": X.dtypes.astype(str),
    "n_missing": X.isna().sum(),
    "missing_ratio": X.isna().mean()
})
miss_df["n_unique"] = [X[c].nunique(dropna=True) for c in X.columns]

miss_df = miss_df.sort_values("missing_ratio", ascending=False)
display(miss_df)

out_csv = Path("reports") / f"{name}_missing_summary.csv"
miss_df.to_csv(out_csv)
print(f"Saved missingness summary to: {out_csv}")

# Quick highlights
all_na = miss_df[miss_df["missing_ratio"] == 1.0].index.tolist()
no_na  = miss_df[miss_df["missing_ratio"] == 0.0].index.tolist()
print(f"Columns fully missing: {len(all_na)} -> {all_na[:10]}{' ...' if len(all_na)>10 else ''}")
print(f"Columns with no missing: {len(no_na)}")


,dtype,n_missing,missing_ratio,n_unique
br_000,float64,49264,0.821067,3806
bq_000,float64,48722,0.812033,4276
bp_000,float64,47740,0.795667,4968
bo_000,float64,46333,0.772217,5838
ab_000,float64,46329,0.772150,29
...,...,...,...,...
ck_000,float64,338,0.005633,45043
ci_000,float64,338,0.005633,45964
cj_000,float64,338,0.005633,7617
bt_000,float64,167,0.002783,45480


Saved missingness summary to: reports/aps_missing_summary.csv
Columns fully missing: 0 -> []
Columns with no missing: 1
